### Общее описание методов работы:

#### Шаг 1. Загрузка данных и базовая подготовка

**Методология:**
Первый этап — загрузка основной агрегированной таблицы и очистка рабочего пространства. Идентификаторы (`id`) и гео-данные читаются строго как строки, чтобы алгоритмы машинного обучения не воспринимали их как количественные признаки и не пытались найти в них числовые закономерности.
Матрица признаков (`X`) и целевой вектор (`y`) разделяются, строки с пропущенным (`NaN`) полом временно откладываются —  их необходимо предсказать на финальном этапе. Отдельно рассматриваются методы оптимизации форматов памяти для ускорения вычислений и снижения потребления RAM при обучении.

#### Шаг 2. Построение ML конвейера и 3-х кратный сплит

**Методология:**
Основная проблема алгоритмов градиентного бустинга при использовании ранней остановки (`early_stopping`) заключается в том, что модель может "подглядывать" в валидационную выборку. Если проверять финальное качество на ней же, результаты будут завышены (утечка данных).

Для получения объективной оценки применяется **3-х кратный сплит**:
1. **Train** — набор для построения и обучения деревьев.
2. **Validation** — набор для контроля ранней остановки (early stopping).
3. **Test** — полностью изолированный набор для финальной оценки модели.

Также используется `ColumnTransformer` для One-Hot кодирования категориальных переменных, так как LightGBM лучше работает с числовыми матрицами. Стадия нормализации пропускается осознанно, поскольку древовидные модели отлично (лучше) стравляются с идентификацией данных без нее.


In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.pipeline import Pipeline
from catboost import CatBoostClassifier, Pool

In [2]:
BASE_DIR = os.getcwd()
DATA_FOLDER = 'data'

#### Выбор модели

In [3]:
file_name = 'df_client_card.csv'
full_path = os.path.join(BASE_DIR, DATA_FOLDER, file_name)

df_client_card = pd.read_csv(full_path)
df_client_card[['id', 'city', 'country']] = df_client_card[['id', 'city', 'country']].astype('str')
df_client_card['education'] = pd.factorize(df_client_card['education'])[0]

total_scores_dict = {}

In [4]:
df_client_card

,id,n_purchases,product_name_n_unique,product_name_top1,product_name_top1_count,product_name_top1_share,product_name_адаптер,product_name_адаптеры,product_name_аквагрим,product_name_аквагримм,...,is_uncertain_mean,is_uncertain_std,adult_vs_kd_ratio,mn_minus_wmn,boys_minus_gls,gender,age,education,city,country
0,0,3,3,стол,1,0.333333,0.0,0.0,0.0,0.0,...,0.000000,0.000000,2.000000,-0.333333,0.000000,0.0,36,0,1201,32
1,100,8,7,кеды,2,0.250000,0.0,0.0,0.0,0.0,...,0.000000,0.000000,8.000000,-0.625000,0.000000,0.0,30,1,1157,32
2,1000,6,4,футболка,2,0.333333,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.333333,0.166667,-0.833333,0.0,39,0,1134,32
3,10000,8,6,толстовка,2,0.250000,0.0,0.0,0.0,0.0,...,0.250000,0.462910,0.285714,-0.125000,0.750000,0.0,45,0,1162,32
4,100000,6,4,футболка,2,0.333333,0.0,0.0,0.0,0.0,...,0.166667,0.408248,7.000000,-1.000000,0.000000,0.0,54,0,1145,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104984,99993,15,5,футболка,5,0.333333,0.0,0.0,0.0,0.0,...,0.333333,0.487950,16.000000,1.000000,0.000000,1.0,45,0,1188,32
104985,99994,2,2,шорты,1,0.500000,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.333333,0.000000,1.000000,1.0,48,0,1178,32
104986,99996,7,6,брюки,2,0.285714,0.0,0.0,0.0,0.0,...,0.000000,0.000000,7.000000,-0.857143,0.000000,0.0,61,0,1167,32
104987,99997,3,3,шорты,1,0.333333,0.0,0.0,0.0,0.0,...,0.333333,0.577350,1.000000,-0.333333,0.333333,1.0,44,0,1197,32


In [5]:
# функция понижения размерности (по оси axis=1) на случай, если модель застрянет в обработке датасета
def x_shrinker(X, shrink_n=10, cols=['product_brand', 'product_name']):
    for col in cols:
        mask = X.columns.str.contains(col)
        cols_to_drop = X[X.columns[mask]].select_dtypes(exclude=['category', 'object']).sum()

        mask = cols_to_drop < shrink_n
        X[f'{col}_other'] = X[cols_to_drop[mask].index].sum(axis=1)
        X = X.drop(columns=cols_to_drop[mask].index)
    return X

In [6]:
# функция вывода оценки после заправки модели
def result_summary(model, X_test, y_test, pr=True):
    proba = model.predict_proba(X_test)[:, 1]
    auc = round(roc_auc_score(y_test, proba), 4)

    pred = (proba >= 0.5).astype(int)
    f1 = round(f1_score(y_test, pred), 4)

    if pr == True:
        print('AUC:\t\t\t', auc)
        print('F1 (th=0.5):\t', f1)
    return {f'{model.__class__.__name__}': [auc, f1]}

#### Исследование LGBMClassifier

In [7]:
# более быстрая версия
mask_unknown = df_client_card['gender'].isna()
X = df_client_card[~mask_unknown].drop(['id', 'gender', 'city', 'country'], axis=1)

#X = X.drop(['product_name_top1', 'product_brand_top1', 'basic_colour_top1'], axis=1)
#X = x_shrinker(X)

y = df_client_card.loc[~mask_unknown, 'gender']
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

# здесь и далее:
# оптимизация памяти: принудительный перевод форматов float64 -> float32 и int64 -> int32
# для ускорения расчетов моделью
f64_cols = X.select_dtypes(include=["float64"]).columns
X[f64_cols] = X[f64_cols].astype("float32")

int64_cols = X.select_dtypes(include=["int64"]).columns
X[int64_cols] = X[int64_cols].astype("int32")

for c in cat_cols:
    X[c] = X[c].astype('category')

# Базовое разбиение: 30% данных - под валидацию и тестирование, 70% - на обучение
X_train, X_fit, y_train, y_fit = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
# Второе разбиение: отложенные 30% делятся пополам на изолированный тест (15%) и валидацию (15%)
X_test, X_val, y_test, y_val = train_test_split(X_fit, y_fit, test_size=0.5, random_state=42, stratify=y_fit)

lgb_fast = LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.12,
    num_leaves=31,
    min_child_samples=300,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_lambda=1.0,
    max_bin=63,
    force_col_wise=True,
    n_jobs=-1,
    random_state=42,
    objective="binary",
    verbose=-1
)

lgb_fast.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    categorical_feature=cat_cols,
    callbacks=[
        early_stopping(stopping_rounds=100, verbose=False),
        #log_evaluation(period=50),
    ]
)

total_scores_dict['LGBMClassifier (fast)'] = result_summary(lgb_fast, X_test, y_test)['LGBMClassifier']

AUC:			 0.8815
F1 (th=0.5):	 0.8464


In [25]:
# более точная версия через настройку гиперпараметров
# и предобработкой категориальных признаков с помощью OHE
mask_unknown = df_client_card['gender'].isna()
X = df_client_card[~mask_unknown].drop(['id', 'gender', 'city', 'country'], axis=1)

#X = X.drop(['product_name_top1', 'product_brand_top1', 'basic_colour_top1'], axis=1)
#X = x_shrinker(X)

y = df_client_card.loc[~mask_unknown, 'gender']
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

f64_cols = X.select_dtypes(include=["float64"]).columns
X[f64_cols] = X[f64_cols].astype("float32")
int64_cols = X.select_dtypes(include=["int64"]).columns
X[int64_cols] = X[int64_cols].astype("int32")

X_train, X_fit, y_train, y_fit = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_fit, y_fit, test_size=0.5, random_state=42, stratify=y_fit)

preprocessor = ColumnTransformer([
        ('cats', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', 'passthrough', num_cols),
    ],
    remainder='drop',
)
# назначение pandas DataFrame вместо numpy array позволяет сохранить оригинальные имена колонок (feature names).
preprocessor.set_output(transform="pandas")
# препроцессор обучается только на X_train, чтобы избежать утечек. Остальные сплиты только трансформируются
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)
X_val = preprocessor.transform(X_val)

lgb = LGBMClassifier(
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary",
    n_jobs=-1,
    random_state=42,
)

lgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        early_stopping(stopping_rounds=200),
        #log_evaluation(200)
    ]
)

total_scores_dict['LGBMClassifier (medium)'] = result_summary(lgb, X_test, y_test)['LGBMClassifier']

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[259]	valid_0's auc: 0.887588	valid_0's binary_logloss: 0.412007
AUC:			 0.8855
F1 (th=0.5):	 0.8489


#### Исследование LogisticRegression

In [9]:
mask_unknown = df_client_card["gender"].isna()

X = df_client_card.loc[~mask_unknown].drop(['id', 'gender', 'city', 'country'], axis=1)
y = df_client_card.loc[~mask_unknown, 'gender'].astype(int)

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.columns.difference(cat_cols).tolist()

# Optional: downcast for memory
f64_cols = X.select_dtypes(include=['float64']).columns
X[f64_cols] = X[f64_cols].astype('float32')
i64_cols = X.select_dtypes(include=['int64']).columns
X[i64_cols] = X[i64_cols].astype('int32')

X_train, X_fit, y_train, y_fit = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_fit, y_fit, test_size=0.5, random_state=42, stratify=y_fit)

preprocessor = ColumnTransformer([
        ('cats', OneHotEncoder(handle_unknown='ignore', sparse_output=True), cat_cols),
        ('num', StandardScaler(), num_cols),
    ],
    remainder='drop',
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)
X_val = preprocessor.transform(X_val)

lr = LogisticRegression(
    solver='saga',
    penalty='l2',
    C=0.3,
    tol=1e-3,
    max_iter=1500,
    random_state=42,
    n_jobs=-1
)

lr.fit(X_train, y_train)

total_scores_dict['LogisticRegression (medium)'] = result_summary(lr, X_test, y_test)['LogisticRegression']

AUC:			 0.8696
F1 (th=0.5):	 0.8448


##### Исследование CatBoostClassifier

In [ ]:
mask_unknown = df_client_card['gender'].isna()
X = df_client_card[~mask_unknown].drop(['id', 'gender', 'city', 'country'], axis=1)
y = df_client_card.loc[~mask_unknown, 'gender']
cat_cols = X.select_dtypes(include='object').columns
cat_idxs = [X.columns.get_loc(col) for col in cat_cols]

X_train, X_fit, y_train, y_fit = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_fit, y_fit, test_size=0.5, random_state=42, stratify=y_fit)

train_pool = Pool(X_train, y_train, cat_features=cat_idxs)
val_pool = Pool(X_val, y_val, cat_features=cat_idxs)
test_pool = Pool(X_test, y_test, cat_features=cat_idxs)

cbc = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    od_type='Iter',
    od_wait=100,
    thread_count=1,
    allow_writing_files=False,
    verbose=200
)

cbc.fit(train_pool, eval_set=val_pool, use_best_model=True)

proba_cbc = cbc.predict_proba(test_pool)[:, 1]
pred_cbc = (proba_cbc > .5) * 1
auc = round(roc_auc_score(y_test, proba_cbc), 4)
f1 = round(f1_score(y_test, pred_cbc), 4)
total_scores_dict['CatBoostClassifier (fast)'] = [auc, f1]

In [11]:
file_name = 'total_scores_dict.pkl'
full_path = os.path.join(BASE_DIR, DATA_FOLDER, file_name)

joblib.dump(total_scores_dict, full_path)
pd.DataFrame.from_dict(total_scores_dict, orient='index', columns=['ROC-AUC', 'F1'])

,ROC-AUC,F1
LGBMClassifier (fast),0.8815,0.8464
LGBMClassifier (medium),0.8855,0.8489
LogisticRegression (medium),0.8696,0.8448
CatBoostClassifier (fast),0.8838,0.8494


**Выбор основной модели:** LGBMClassifier (medium) и CatBoostClassifier (fast) показали примерно одинаковые результаты при скоринге, но LGBMClassifier работает гораздо быстрее, в этой связи она принимается за основную модель.

In [26]:
file_name = 'lgb_reference.pkl'
full_path = os.path.join(BASE_DIR, DATA_FOLDER, file_name)

joblib.dump(lgb, full_path)

['/Users/user/PycharmProjects/test/sport/data/lgb_reference.pkl']

### Шаг 3. Проверка стабильности основной модели через кросс-валидацию (Stratified K-Fold)

**Методология:**
Разовое разбиение выборки во многом зависит от случайности конкретного сплита. Чтобы получить оценить устойчивость и постоянство предсказаний (по дисперсии дисперсию метрик для каждого фолда), применяется кросс-валидация.
Внутри ручного цикла `StratifiedKFold` строго сохраняется логика **3-х кратного сплита**: каждый фолд делится на обучение, внутреннюю валидацию и тест. Это гарантирует, что метрики каждого фолда получены на абсолютно незнакомых модели данных.


In [ ]:
# словарь будущих оценок для каждого фолда кросс валидации
cv = {
    'test_roc_auc': np.array([]),
    'test_f1': np.array([])
}

mask_unknown = df_client_card['gender'].isna()
X = df_client_card[~mask_unknown].drop(['id', 'gender', 'city', 'country'], axis=1)

#X = X.drop(['product_name_top1', 'product_brand_top1', 'basic_colour_top1'], axis=1)
#X = x_shrinker(X)

y = df_client_card.loc[~mask_unknown, 'gender']
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

f64_cols = X.select_dtypes(include=["float64"]).columns
X[f64_cols] = X[f64_cols].astype("float32")
int64_cols = X.select_dtypes(include=["int64"]).columns
X[int64_cols] = X[int64_cols].astype("int32")

# применение StratifiedKFold гарантирует сохранение баланса классов (М/Ж) внутри каждого из 5 фолдов
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


for i, (fit_ind, test_ind) in enumerate(skf.split(X, y)):

    X_fit, y_fit = X.iloc[fit_ind], y.iloc[fit_ind]
    X_test, y_test = X.iloc[test_ind], y.iloc[test_ind]

    X_train, X_val, y_train, y_val = train_test_split(X_fit, y_fit, test_size=0.15, random_state=42, stratify=y_fit)

    preprocessor = ColumnTransformer([
            ('cats', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
            ('num', 'passthrough', num_cols),
        ],
        remainder='drop',
    )

    preprocessor.set_output(transform="pandas")

    X_train = preprocessor.fit_transform(X_train)
    X_test = preprocessor.transform(X_test)
    X_val = preprocessor.transform(X_val)

    lgb = LGBMClassifier(
        n_estimators=5000,
        learning_rate=0.03,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective="binary",
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )

    lgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[
            early_stopping(stopping_rounds=200),
        ]
    )

    # получение результатов и сохранение в словарь оценок
    result = result_summary(lgb, X_test, y_test)['LGBMClassifier']
    cv['test_roc_auc'] = np.append(cv['test_roc_auc'], result[0])
    cv['test_f1'] = np.append(cv['test_f1'], result[1])
    print('\n')


In [24]:
print(f'Среднеквадратическое отклонение по оценкам:\n'
      f'ROC-AUC: {cv['test_roc_auc'].std().round(4)}\n'
      f'F1: {cv['test_f1'].std().round(4)}\n'
      f'Не превышает 0.03\n'
      f'Модель стабильна в предиктах'
      )

Среднеквадратическое отклонение по оценкам:
ROC-AUC: 0.0029
F1: 0.0018
Не превышает 0.03
Модель стабильна в предиктах


In [ ]:
# сохранение словаря кросс валидации
file_name = 'cv_lgb.pkl'
full_path = os.path.join(BASE_DIR, DATA_FOLDER, file_name)

joblib.dump(cv, full_path)

### Шаг 4. Локальный отбор признаков

**Методология:**
Датасет собран из множества транзакционных агрегатов (по бренду, цвету, времени и т.д.), ниже проверяется их полезность., итеративно исключая из обучения целые смысловые группы признаков.
Это своеобразная ручная оценка важности (feature selection): если при удалении конкретной группы метрика на тестовой выборке падает в сравнении с базовой, значит эти признаки полезны. Если метрика растет или остается неизменной — группа вносит только 'шум' или переобучает модель.


In [15]:
# словарь с логическими группами признаков для экспериментов (по названию, бренду, цвету и т.д.)
doubtful_feature_combinations = {
    'week_all': ['day_of_week_1', 'day_of_week_2',
                     'day_of_week_3', 'day_of_week_4', 'day_of_week_5', 'day_of_week_6',
                     'day_of_week_7'],
    'week_top1': ['day_of_week_top1_count',
                     'day_of_week_top1_share'],
    'week_n_unique': ['day_of_week_n_unique', 'day_of_week_entropy'],
    'product_name_top1': ['product_name_top1', 'product_name_top1_count',
                     'product_name_top1_share'],
    'product_name_n_unique': ['product_name_n_unique', 'product_name_entropy'],
    'product_brand_top1': ['product_brand_top1_count', 'product_brand_top1_share'],
    'product_brand_n_unique': ['product_brand_n_unique', 'product_brand_entropy'],
    'basic_colour_top1': ['basic_colour_top1_count',
                     'basic_colour_top1_share'],
    'basic_colour_n_unique': ['basic_colour_n_unique', 'basic_colour_entropy'],
    'prod_gender': ['prod_kids_sum', 'boys_minus_gls', 'prod_kids_mean'],
    'modificstors_mean': ['is_bright_mean', 'is_dark_mean',
                     'is_iridescent_mean', 'is_iridescent_std', 'is_light_mean', 'is_multicolored_mean',
                     'is_neutral_mean', 'is_pastel_mean',
                     'is_rare_mean', 'is_saturated_mean',
                     'is_texture_mean', 'is_uncertain_mean'],
    'modificstors_std': ['is_bright_std', 'is_dark_std', 'is_light_std', 'is_multicolored_std',
                     'is_neutral_std', 'is_pastel_std', 'is_rare_std', 'is_saturated_std', 'is_texture_std',
                     'is_uncertain_std'],
    'top_1': ['day_of_week_top1', 'product_name_top1', 'product_brand_top1']
}

In [16]:
feat_group_scores = {}
for feat_group in doubtful_feature_combinations:

    mask_unknown = df_client_card['gender'].isna()
    X = df_client_card[~mask_unknown].drop(['id', 'gender', 'city', 'country'], axis=1)

    X = X.drop(doubtful_feature_combinations[feat_group], axis=1)

    y = df_client_card.loc[~mask_unknown, 'gender']
    cat_cols = X.select_dtypes(include='object').columns.tolist()
    num_cols = X.select_dtypes(exclude='object').columns.tolist()

    f64_cols = X.select_dtypes(include=["float64"]).columns
    X[f64_cols] = X[f64_cols].astype("float32")
    int64_cols = X.select_dtypes(include=["int64"]).columns
    X[int64_cols] = X[int64_cols].astype("int32")

    X_train, X_fit, y_train, y_fit = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_test, X_val, y_test, y_val = train_test_split(X_fit, y_fit, test_size=0.5, random_state=42, stratify=y_fit)

    preprocessor = ColumnTransformer([
            ('cats', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
            ('num', 'passthrough', num_cols),
        ],
        remainder='drop',
    )

    preprocessor.set_output(transform="pandas")

    X_train = preprocessor.fit_transform(X_train)
    X_test = preprocessor.transform(X_test)
    X_val = preprocessor.transform(X_val)

    lgb = LGBMClassifier(
        n_estimators=5000,
        learning_rate=0.03,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective="binary",
        n_jobs=-1,
        random_state=42,
    )

    lgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[
            early_stopping(stopping_rounds=200),
            #log_evaluation(200)
        ]
    )

    # оценки сохраняются в словарь оценок feat_group_scores для дальнейшего сравнения
    feat_group_scores[f'{feat_group}'] = result_summary(lgb, X_test, y_test)['LGBMClassifier']
    print('\n')
feat_group_scores['REFERENCE'] = total_scores_dict['LGBMClassifier (medium)']

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[250]	valid_0's auc: 0.887443	valid_0's binary_logloss: 0.412088
AUC:			 0.8853
F1 (th=0.5):	 0.849


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[253]	valid_0's auc: 0.886927	valid_0's binary_logloss: 0.412802
AUC:			 0.8852
F1 (th=0.5):	 0.8493


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[233]	valid_0's auc: 0.887016	valid_0's binary_logloss: 0.412699
AUC:			 0.8851
F1 (th=0.5):	 0.8498


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[289]	valid_0's auc: 0.887193	valid_0's binary_logloss: 0.412265
AUC:			 0.8857
F1 (th=0.5):	 0.8487


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[224]	valid_0's auc: 0.887253	valid_0's binary_logloss: 0.412436
AUC:			 0.8855
F1 (th=0.5):	 0.8491


T

In [17]:
file_name = 'feat_group_scores.pkl'
full_path = os.path.join(BASE_DIR, DATA_FOLDER, file_name)

joblib.dump(feat_group_scores, full_path)
joblib.load(full_path)

{'week_all': [0.8853, 0.849],
 'week_top1': [0.8852, 0.8493],
 'week_n_unique': [0.8851, 0.8498],
 'product_name_top1': [0.8857, 0.8487],
 'product_name_n_unique': [0.8855, 0.8491],
 'product_brand_top1': [0.885, 0.8488],
 'product_brand_n_unique': [0.8855, 0.8496],
 'basic_colour_top1': [0.8857, 0.8486],
 'basic_colour_n_unique': [0.8852, 0.8491],
 'prod_gender': [0.8856, 0.8495],
 'modificstors_mean': [0.8854, 0.8491],
 'modificstors_std': [0.8854, 0.8499],
 'top_1': [0.8856, 0.8491],
 'REFERENCE': [0.8855, 0.8489]}

In [18]:
print('Итоговые результаты проверки признаков и анализ их рентабельности:')
df_doubt = pd.DataFrame.from_dict(feat_group_scores, orient='index', columns=['ROC-AUC', 'F1'])
df_doubt

Итоговые результаты проверки признаков и анализ их рентабельности:


,ROC-AUC,F1
week_all,0.8853,0.8490
week_top1,0.8852,0.8493
week_n_unique,0.8851,0.8498
product_name_top1,0.8857,0.8487
product_name_n_unique,0.8855,0.8491
product_brand_top1,0.8850,0.8488
product_brand_n_unique,0.8855,0.8496
basic_colour_top1,0.8857,0.8486
basic_colour_n_unique,0.8852,0.8491
prod_gender,0.8856,0.8495


In [19]:
# признаки, понижающие метрики скоринга
mask = ((df_doubt['ROC-AUC'] > df_doubt.loc['REFERENCE', 'ROC-AUC'])
    & (df_doubt['F1'] > df_doubt.loc['REFERENCE', 'F1'])
)
df_doubt[mask]

,ROC-AUC,F1
prod_gender,0.8856,0.8495
top_1,0.8856,0.8491


In [20]:
# повторный скоринг изменений с удаленными сочетаниями понижающих признаков (prod_gender и top_1)
col_to_drop_additionally = [f for k in df_doubt[mask].index for f in doubtful_feature_combinations[k]]

mask_unknown = df_client_card['gender'].isna()
X = df_client_card[~mask_unknown].drop(col_to_drop_additionally + ['id', 'gender', 'city', 'country'], axis=1)

y = df_client_card.loc[~mask_unknown, 'gender']
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

f64_cols = X.select_dtypes(include=["float64"]).columns
X[f64_cols] = X[f64_cols].astype("float32")
int64_cols = X.select_dtypes(include=["int64"]).columns
X[int64_cols] = X[int64_cols].astype("int32")

X_train, X_fit, y_train, y_fit = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_fit, y_fit, test_size=0.5, random_state=42, stratify=y_fit)

preprocessor = ColumnTransformer([
        ('cats', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', 'passthrough', num_cols),
    ],
    remainder='drop',
)

preprocessor.set_output(transform="pandas")

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)
X_val = preprocessor.transform(X_val)

lgb_temp = LGBMClassifier(
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary",
    n_jobs=-1,
    random_state=42,
)

lgb_temp.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        early_stopping(stopping_rounds=200),
        #log_evaluation(200)
    ]
)

result_summary(lgb_temp, X_test, y_test)['LGBMClassifier']

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[235]	valid_0's auc: 0.887128	valid_0's binary_logloss: 0.412859
AUC:			 0.8855
F1 (th=0.5):	 0.8486


[0.8855, 0.8486]

Итоговые результаты проверки признаков и анализ их рентабельности показывают, что все признаки можно оставить для обучения

In [ ]:
file_name = 'lgb.pkl'
full_path = os.path.join(BASE_DIR, DATA_FOLDER, file_name)

joblib.dump(lgb, file_name)
lgb = joblib.load(file_name)

### Шаг 5. Поиск пороговой границы вероятности (отсев порога `threshold`)

**Методология:**
Поскольку классы в задаче могут иметь дисбаланс, стандартная граница отсечения вероятности (probability > 0.5) часто бывает неэффективной. Чтобы максимизировать баланс между Precision (точностью) и Recall (полнотой), перебираются различные пороги (`threshold`) и выбирается тот, который дает абсолютный максимум метрики **F1-Score**.


In [33]:
# устанавливается диапазон границ, которые будут перебираться в цикле

# загрузка референса
file_name = 'lgb_reference.pkl'
full_path = os.path.join(BASE_DIR, DATA_FOLDER, file_name)
lgb = joblib.load(full_path)


threshold_list = np.arange(.35, .75, .0001)
proba = lgb.predict_proba(X_test)[:, 1]

f1_dict = {}
for t in threshold_list:
    pred = (proba > t) * 1
    f1_dict[f'{t:.4f}'] = f1_score(y_test, pred)

# составление результирующего датасета порогов и их значений
df_f1 = pd.DataFrame.from_dict(f1_dict, orient='index', columns=['f1']).round(5)

# поиск максимума для датасета порогов с помощью numpy функции np.argwhere
mx = df_f1.values.max()
max_f1_array = np.argwhere(df_f1.values == mx)
for r, c in max_f1_array:
    print(f'Граница разделения (threshold) для {df_f1.columns[c].upper()}: {df_f1.index[r]}\n'
          f'дает оценку {mx}')

Граница разделения (threshold) для F1: 0.4013
дает оценку 0.85304




### Шаг 6. Обучение на вторичных коэффициентах

Это альтернативное исследование, подразумевающее наличие внутренних коэффициентов компании, рассчитанных для каждого клиента. В качестве исходного условия подразумевается, что коэффициент может быть только один.

**Методология:**
Следующая итерация модели строится исключительно на предварительно рассчитанных бизнес-коэффициентах (`df_coeffs`).

In [34]:
file_name = 'coeffs.csv'
full_path = os.path.join(BASE_DIR, DATA_FOLDER, file_name)

df_coeffs = pd.read_csv(full_path, dtype={'id': 'str'})
df_coeffs

,id,lbt_coef,ac_coef,sm_coef,personal_coef
0,0,5.078678,-0.307147,0.959027,0.5072
1,3,7.764766,-0.030225,0.794720,0.4304
2,4,4.569378,0.063693,0.820892,0.5072
3,6,8.150379,0.075200,0.836140,0.4304
4,7,5.188231,-0.000134,0.944113,0.5072
...,...,...,...,...,...
104984,177998,4.740988,0.364797,1.165888,0.5072
104985,177999,7.303172,0.431899,1.317100,0.4304
104986,178001,5.241579,0.430391,0.356182,0.2576
104987,178002,7.542436,-0.290921,0.800338,0.4304


In [36]:
df_coeffs

,id,lbt_coef,ac_coef,sm_coef,personal_coef
0,0,5.078678,-0.307147,0.959027,0.5072
1,3,7.764766,-0.030225,0.794720,0.4304
2,4,4.569378,0.063693,0.820892,0.5072
3,6,8.150379,0.075200,0.836140,0.4304
4,7,5.188231,-0.000134,0.944113,0.5072
...,...,...,...,...,...
104984,177998,4.740988,0.364797,1.165888,0.5072
104985,177999,7.303172,0.431899,1.317100,0.4304
104986,178001,5.241579,0.430391,0.356182,0.2576
104987,178002,7.542436,-0.290921,0.800338,0.4304


In [37]:
# создание словаря для хранения данных об оценках для каждой итерации
feat_group_scores = {}

# составление списка коэффициентов
doubtful_feature_combinations = df_coeffs.drop(columns=['id']).columns

# перебор каждого коэффициента по-отдельности в сочетании с основными признаками
for feat_group in doubtful_feature_combinations:

    mask_unknown = df_client_card['gender'].isna()

    X = pd.merge(df_client_card, df_coeffs[['id', feat_group]], on='id', how='left')[~mask_unknown].drop(['id', 'gender', 'city', 'country'], axis=1)

    #X = X.drop(doubtful_feature_combinations[feat_group], axis=1)

    y = df_client_card.loc[~mask_unknown, 'gender']
    cat_cols = X.select_dtypes(include='object').columns.tolist()
    num_cols = X.select_dtypes(exclude='object').columns.tolist()

    f64_cols = X.select_dtypes(include=["float64"]).columns
    X[f64_cols] = X[f64_cols].astype("float32")
    int64_cols = X.select_dtypes(include=["int64"]).columns
    X[int64_cols] = X[int64_cols].astype("int32")

    X_train, X_fit, y_train, y_fit = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_test, X_val, y_test, y_val = train_test_split(X_fit, y_fit, test_size=0.5, random_state=42, stratify=y_fit)

    preprocessor = ColumnTransformer([
            ('cats', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
            ('num', 'passthrough', num_cols),
        ],
        remainder='drop',
    )

    preprocessor.set_output(transform="pandas")

    X_train = preprocessor.fit_transform(X_train)
    X_test = preprocessor.transform(X_test)
    X_val = preprocessor.transform(X_val)

    lgb = LGBMClassifier(
        n_estimators=5000,
        learning_rate=0.03,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective="binary",
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )

    lgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[
            early_stopping(stopping_rounds=200),
            #log_evaluation(200)
        ]
    )
    print('='*30)
    print(feat_group)
    feat_group_scores[f'{feat_group}'] = result_summary(lgb, X_test, y_test)['LGBMClassifier']
    print('='*30)

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[479]	valid_0's auc: 1	valid_0's binary_logloss: 0.000588742
lbt_coef
AUC:			 1.0
F1 (th=0.5):	 1.0
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[276]	valid_0's auc: 0.887397	valid_0's binary_logloss: 0.411955
ac_coef
AUC:			 0.8856
F1 (th=0.5):	 0.8499
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[315]	valid_0's auc: 0.900571	valid_0's binary_logloss: 0.387138
sm_coef
AUC:			 0.898
F1 (th=0.5):	 0.8583
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[155]	valid_0's auc: 1	valid_0's binary_logloss: 0.00891617
personal_coef
AUC:			 1.0
F1 (th=0.5):	 0.9999


In [38]:
pd.DataFrame.from_dict(feat_group_scores, orient='index', columns=['ROC-AUC', 'F1'])

,ROC-AUC,F1
lbt_coef,1.0000,1.0000
ac_coef,0.8856,0.8499
sm_coef,0.8980,0.8583
personal_coef,1.0000,0.9999


In [47]:
# анализ корреляции коэффициентов показывает,
# что в целом для предсказания достаточно было бы использовать только данные о коэффициентах
# здесь этот вариант исследуется лишь в конце,
# чтобы показать навыки возможной работы с более неустойчивыми данными для определения классов предсказания
coef_corr = pd.merge(df_client_card[['id', 'gender']], df_coeffs, on='id', how='left')
mask = coef_corr['gender'].notna()
coef_corr[mask].corr().round(2)

,id,gender,lbt_coef,ac_coef,sm_coef,personal_coef
id,1.0,0.00,0.00,0.00,0.00,-0.00
gender,0.0,1.00,0.93,0.00,-0.12,-0.61
lbt_coef,0.0,0.93,1.00,0.00,-0.09,-0.47
ac_coef,0.0,0.00,0.00,1.00,0.01,0.05
sm_coef,0.0,-0.12,-0.09,0.01,1.00,0.21
personal_coef,-0.0,-0.61,-0.47,0.05,0.21,1.00


In [40]:
coef_corr

,id,gender,lbt_coef,ac_coef,sm_coef,personal_coef
0,0,0.0,5.078678,-0.307147,0.959027,0.5072
1,100,0.0,4.520433,0.163553,1.293888,0.5584
2,1000,0.0,4.821220,0.141673,0.794559,0.5072
3,10000,0.0,5.661616,-0.052453,0.712793,0.5072
4,100000,0.0,5.721683,0.451163,1.019668,0.5072
...,...,...,...,...,...,...
104984,99993,1.0,7.618278,0.098465,1.215570,0.4304
104985,99994,1.0,7.899935,0.520464,0.741710,0.4304
104986,99996,0.0,5.752846,0.573485,1.168146,0.5072
104987,99997,1.0,8.271815,-0.305303,0.877624,0.4304


In [58]:
# подтверждение гипотезы автономного использования коэффициентов
mask_unknown = coef_corr['gender'].isna()
X = coef_corr[~mask_unknown].drop(['id', 'gender'], axis=1)
y = coef_corr.loc[~mask_unknown, 'gender']

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

X_train, X_fit, y_train, y_fit = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_fit, y_fit, test_size=0.5, random_state=42, stratify=y_fit)

preprocessor = ColumnTransformer([
        ('cats', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', 'passthrough', num_cols),
    ],
    remainder='drop',
)

preprocessor.set_output(transform="pandas")

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)
X_val = preprocessor.transform(X_val)

lgb = LGBMClassifier(
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary",
    n_jobs=-1,
    random_state=42,
    verbose=-1
)

lgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        early_stopping(stopping_rounds=200),
        #log_evaluation(200)
    ]
)

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[92]	valid_0's auc: 1	valid_0's binary_logloss: 0.0320328


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.03
,n_estimators,5000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,50


In [56]:
# Итоговые оценки близки к 100%
result_summary(lgb, X_test, y_test)

AUC:			 1.0
F1 (th=0.5):	 0.9999


{'LGBMClassifier': [1.0, 0.9999]}

In [57]:
# итоговый поиск значения оптимального порога вероятности предсказания
threshold_list = np.arange(.35, .75, .01)
proba = lgb.predict_proba(X_test)[:, 1]

f1_dict = {}
for t in threshold_list:
    pred = (proba > t) * 1
    f1_dict[f'{t:.4f}'] = f1_score(y_test, pred)

df_f1 = pd.DataFrame.from_dict(f1_dict, orient='index', columns=['f1'])
mx = df_f1.values.max()
max_f1_array = np.argwhere(df_f1.values == mx)
for r, c in max_f1_array:
    print(f'Граница разделения (threshold) для {df_f1.columns[c].upper()}: {df_f1.index[r]}\n'
          f'дает оценку {mx}')

Граница разделения (threshold) для F1: 0.4700
дает оценку 0.9999348152010951
Граница разделения (threshold) для F1: 0.4800
дает оценку 0.9999348152010951
Граница разделения (threshold) для F1: 0.4900
дает оценку 0.9999348152010951
Граница разделения (threshold) для F1: 0.5000
дает оценку 0.9999348152010951
Граница разделения (threshold) для F1: 0.5100
дает оценку 0.9999348152010951
Граница разделения (threshold) для F1: 0.5200
дает оценку 0.9999348152010951
Граница разделения (threshold) для F1: 0.5300
дает оценку 0.9999348152010951
Граница разделения (threshold) для F1: 0.5400
дает оценку 0.9999348152010951
Граница разделения (threshold) для F1: 0.5500
дает оценку 0.9999348152010951
Граница разделения (threshold) для F1: 0.5600
дает оценку 0.9999348152010951
Граница разделения (threshold) для F1: 0.5700
дает оценку 0.9999348152010951
Граница разделения (threshold) для F1: 0.5800
дает оценку 0.9999348152010951
Граница разделения (threshold) для F1: 0.5900
дает оценку 0.9999348152010951

### Шаг 6. Финальное обучение и заполнение пропусков

**Методология:**
После того как оптимальные гиперпараметры модели конфигурации и пороговые значения (`threshold`) найдены, чистое тестовое множество (`X_test`) больше не требуется. Для максимизации обобщающей способности модель переобучается на всех доступных размеченных данных (за исключением доли на валидацию для ранней остановки `early_stopping`).
Обученная модель применяется к строкам с неизвестным полом (`mask_unknown`) для заполнения. Заполненная таблица сохраняется для дальнейших анализов.


In [60]:
# подтверждение гипотезы автономного использования коэффициентов
mask_unknown = coef_corr['gender'].isna()
X = coef_corr[~mask_unknown].drop(['id', 'gender'], axis=1)
y = coef_corr.loc[~mask_unknown, 'gender']

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
        ('cats', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', 'passthrough', num_cols),
    ],
    remainder='drop',
)

preprocessor.set_output(transform="pandas")

X_train = preprocessor.fit_transform(X_train)
X_val = preprocessor.transform(X_val)

lgb = LGBMClassifier(
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary",
    n_jobs=-1,
    random_state=42,
    verbose=-1
)

lgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        early_stopping(stopping_rounds=200),
        #log_evaluation(200)
    ]
)

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[78]	valid_0's auc: 1	valid_0's binary_logloss: 0.0487919


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.03
,n_estimators,5000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,50


In [65]:
mask = df_f1['f1'] == df_f1['f1'].max()

th = df_f1[mask].index.astype(float).values.mean()
print(f'threshold = {th:.4f}')
X_nan = df_client_card[mask_unknown].drop(['id', 'gender', 'city', 'country'], axis=1)

f64_cols = X_nan.select_dtypes(include=["float64"]).columns
X_nan[f64_cols] = X_nan[f64_cols].astype("float32")
int64_cols = X_nan.select_dtypes(include=["int64"]).columns
X_nan[int64_cols] = X_nan[int64_cols].astype("int32")

X_nan = preprocessor.transform(X_nan)
y_nan = (lgb.predict_proba(X_nan)[:, 1] > th).astype(int)

# Сохраняем и в основную таблицу, и в таблицу коэффициентов
df_client_card.loc[mask_unknown, 'gender'] = y_nan
coef_corr.loc[mask_unknown, 'gender'] = y_nan

file_name = 'df_coeffs_filled.csv'
full_path = os.path.join(BASE_DIR, DATA_FOLDER, file_name)
coef_corr.to_csv(full_path, index=False)
print(f'Coefficients results saved in {full_path}')

file_name2 = 'df_client_card_filled.csv'
full_path2 = os.path.join(BASE_DIR, DATA_FOLDER, file_name2)
df_client_card.to_csv(full_path2, index=False)
print(f'Client cards results saved in {full_path2}')


threshold = 0.5450
Coefficients results saved in /Users/user/PycharmProjects/test/sport/data/df_coeffs_filled.csv
Client cards results saved in /Users/user/PycharmProjects/test/sport/data/df_client_card_filled.csv


In [66]:
file_name = 'lgb_best.pkl'
full_path = os.path.join(BASE_DIR, DATA_FOLDER, file_name)

joblib.dump(lgb, full_path)

['/Users/user/PycharmProjects/test/sport/data/lgb_best.pkl']

In [67]:
coef_corr

,id,gender,lbt_coef,ac_coef,sm_coef,personal_coef
0,0,0.0,5.078678,-0.307147,0.959027,0.5072
1,100,0.0,4.520433,0.163553,1.293888,0.5584
2,1000,0.0,4.821220,0.141673,0.794559,0.5072
3,10000,0.0,5.661616,-0.052453,0.712793,0.5072
4,100000,0.0,5.721683,0.451163,1.019668,0.5072
...,...,...,...,...,...,...
104984,99993,1.0,7.618278,0.098465,1.215570,0.4304
104985,99994,1.0,7.899935,0.520464,0.741710,0.4304
104986,99996,0.0,5.752846,0.573485,1.168146,0.5072
104987,99997,1.0,8.271815,-0.305303,0.877624,0.4304
